In [9]:
import pandas as pd

In [ ]:
class BacktestCrossMA:
    def __init__(self) -> None:
        self.df = pd.DataFrame()
        self.days = []
        self.trades = []

    def load_data(self, path):
        self.df = pd.read_csv(path)
        self.df["date"] = pd.to_datetime(self.df["date"], unit="ms")
        self.df = self.df.set_index(self.df["date"])
        del self.df["date"]

    def populate_indicators(self):
        self.df['ma100'] = self.df['close'].rolling(100).mean()
        self.df['ma200'] = self.df['close'].rolling(200).mean()

    def populate_signals(self):
        self.df['buy_signal'] = False
        self.df['sell_signal'] = False
        self.df.loc[(self.df['ma100'] > self.df['ma200']), 'buy_signal'] = True
        self.df.loc[(self.df['ma100'] < self.df['ma200']), 'sell_signal'] = True

    def run_backtest(self):
        balance = 1000
        position = None
        asset = "BTC"
        fee = 0.001
        self.days = []
        self.trades = []
        previous_day = -1

        for index, row in self.df.iterrows():
        
            current_day = index.day
            if previous_day != current_day:
                temp_balance = balance
                if position:
                    close_price = row["close"]
                    trade_result = (close_price - position["open_price"]) / position[
                        "open_price"
                    ]
                    close_size = (
                        position["open_size"] + position["open_size"] * trade_result
                    )
                    fees = close_size * fee
                    close_size = close_size - fees
                    temp_balance = temp_balance + close_size - position["open_size"]
                self.days.append({
                    "day":index.date(),
                    "balance":temp_balance,
                    "price":row['close']
                })
            previous_day = current_day

            if position is None and row['buy_signal']:
                open_price = row['close']
                open_size = balance
                fees = fee * open_size
                open_size -= fees
                balance -= fees 
                position = {
                    'open_price': open_price,
                    'open_size': open_size,
                }
                print(f"{index} - Buy for {balance}$ of {asset} at {open_price}$, fees {fees}$")

            elif position and row['sell_signal']:
                close_price = row['close']
                trade_result = (close_price - position['open_price']) / position['open_price']
                close_size = position['open_size'] + position['open_size'] * trade_result
                fees = fee * close_size
                close_size -= fees
                balance = balance + close_size - position['open_size']
                position = None
                print(f"{index} - Sell for {balance}$ of {asset} at {close_price}$, fees {fees}$")

        print(f"Final balance: {balance}$")

bt = BacktestCrossMA()
bt.load_data('BTC-USDT.csv')
bt.populate_indicators()
bt.populate_signals()
bt.run_backtest()

In [12]:
pd.DataFrame(bt.days)

,day,balance,price
0,2017-08-17,1000.000000,4308.83
1,2017-08-18,1000.000000,4286.53
2,2017-08-19,1000.000000,4163.49
3,2017-08-20,1000.000000,4086.09
4,2017-08-21,1000.000000,4108.28
...,...,...,...
2142,2023-06-29,9780.490882,30079.99
2143,2023-06-30,9890.180034,30441.34
2144,2023-07-01,9601.443386,30469.83
2145,2023-07-02,9601.443386,30575.77
